# 02 · Evaluate — SOC Alert-Triage Assistant

**Vertical:** Cybersecurity · **Use case:** alert-triage · **ADLC stage:** Evaluate

> Self-directed module · ~15 min. By the end you can:
> - push and preview a 10-example SOC triage dataset in LangSmith
> - write two visible evaluators — an LLM-judge (severity) + a deterministic completeness check
> - dry-run one example, scale with `evaluate()`, read a scorecard, then iterate & compare experiments
>
> 🧭 **Builds on Lab 01; runs standalone.** Tools + prompt are Lab 01's (imported); what's shown here is
> the visible `create_agent` eval target and the evaluator rubrics.

Lab 01 built the agent. Now we measure whether its triage reports are accurate, grounded, safe, and
structured enough for an analyst handoff.

![Architecture](images/02_evaluate.png)

## 0. Setup

Thin here — this lab only needs env loading, imports from the content root, and the active workspace.

In [ ]:
import json
import os
import sys
from pathlib import Path

from dotenv import find_dotenv, load_dotenv

load_dotenv(find_dotenv(), override=True)  # lab .env wins over a globally-set key/workspace
if "gateway.smith.langchain.com" in os.environ.get("OPENAI_BASE_URL", ""):
    os.environ.pop("OPENAI_BASE_URL", None)  # use the provider key directly, not an injected gateway

_root = Path.cwd()
while _root != _root.parent and not (_root / "utils").is_dir():
    _root = _root.parent
sys.path.insert(0, str(_root))

print("workspace:", os.getenv("LANGSMITH_WORKSPACE_ID"), "| tracing:", os.getenv("LANGSMITH_TRACING"))

## 1. Rebuild the agent — the eval target

We grade what we'd ship: the same `create_agent` wiring from Lab 01. Tools + prompt are imported as
plumbing; the agent construction stays visible and is reused by the eval target.

In [ ]:
from langchain.agents import create_agent

from alert_triage_graph import AlertTriageReport, PROMPT_NAME, PROMPT_PATH, alert_triage_tools
from utils.models import get_model
from utils.prompts import dataset_url, get_prompt

model = get_model()
tools = alert_triage_tools()  # get_alert, get_asset_context, search_incident_history, search_soc_kb
system_prompt = get_prompt(PROMPT_NAME, fallback=PROMPT_PATH.read_text().strip())

agent = create_agent(
    model=model,
    tools=tools,
    system_prompt=system_prompt,
    response_format=AlertTriageReport,  # typed report (schema taught in Lab 01), imported as plumbing here
)

**The agent's shape** — render the compiled graph before evaluating it.

In [ ]:
from IPython.display import Image, display

display(Image(agent.get_graph().draw_mermaid_png()))

## 2. Evaluate on the alert-triage dataset

`alert-triage-eval` is committed in `evals/`: 10 synthetic alert questions with reference severity,
disposition, containment, expected tools, and a minimal correct report.

> ⚠️ `evaluate()` grades the **remote** dataset, not your local JSON. If fixtures or reference answers
> changed, run `seed_dataset(refresh=True)` so judges don't compare against stale references.

In [ ]:
from evals.seed_alert_triage import DATASET_NAME, DATASET_PATH, seed_dataset

seed_dataset()  # idempotent push to LangSmith; pass refresh=True after editing the JSON
print("📊 dataset:", dataset_url(DATASET_NAME))  # workspace-scoped — open it before running

In [ ]:
example = json.loads(DATASET_PATH.read_text())[0]
print("Q         :", example["inputs"]["messages"][-1]["content"])
print("reference :", example["outputs"]["reference_answer"].splitlines()[:6])
print("tools     :", example["outputs"]["expected_tools"])

**The eval target** wraps the §1 `agent` — one invoke per example. `response_format` means the report
comes back as a validated `AlertTriageReport` in `structured_response`; we serialize it to JSON so the
judges below grade a stable, complete report.

In [ ]:
def run_agent(inputs: dict) -> dict:
    result = agent.invoke({"messages": inputs["messages"]})  # same agent object we rendered above
    report = result["structured_response"]  # validated AlertTriageReport, not free text
    return {"answer": report.model_dump_json(indent=2)}

## 3. The evaluators

We keep the two most representative — one of each kind — so the core scorecard stays readable. Each
returns `{"key", "score", "comment"}`. (Need more? §7 points to LangSmith's evaluator template catalog.)

| Evaluator | Kind | Receives | Checks |
|---|---|---|---|
| `severity_correctness` | reference-based **LLM judge** | `inputs, outputs, reference_outputs` | severity, disposition, escalation |
| `report_completeness_schema` | **deterministic** (trace-based) | `run` tree | required sections + tool-grounded behavior |

### Evaluator #1: severity_correctness

Severity is not the same as disposition: a blocked privileged-identity attack can still be High.

In [ ]:
from pydantic import BaseModel, Field

SEVERITY_CORRECTNESS_PROMPT = """You grade a SOC alert-triage assistant's final answer against a reference triage.

Return True if the answer gets the core decision right: severity, disposition, and escalation flag.
Allow correct additional details beyond the reference. Fail for wrong severity, wrong disposition, wrong
escalation decision, contradiction, or invented facts that change the decision.

<question>{question}</question>
<answer>{answer}</answer>
<reference_answer>{reference}</reference_answer>
<expected_severity>{severity}</expected_severity>
<expected_disposition>{disposition}</expected_disposition>
<expected_escalate>{escalate}</expected_escalate>"""


class SeverityCorrectnessScore(BaseModel):
    reasoning: str = Field(description="Brief reasoning about severity, disposition, and escalation.")
    score: bool = Field(description="True if the core triage decision is correct.")


def severity_correctness_evaluator(inputs: dict, outputs: dict, reference_outputs: dict) -> dict:
    judge = get_model().with_structured_output(SeverityCorrectnessScore)
    v = judge.invoke(SEVERITY_CORRECTNESS_PROMPT.format(
        question=inputs["messages"][-1]["content"],
        answer=outputs["answer"],
        reference=reference_outputs["reference_answer"],
        severity=reference_outputs["severity"],
        disposition=reference_outputs["disposition"],
        escalate=reference_outputs["escalate"],
    ))
    return {"key": "severity_correctness", "score": v.score, "comment": v.reasoning}

### Evaluator #2: report_completeness_schema

Trace-based: it walks `run.child_runs`, counts actual tool calls, and checks that the final report has
the SOC handoff sections the prompt requires.

In [ ]:
from langsmith.schemas import Run


def report_completeness_schema_evaluator(run: Run) -> dict:
    def _tool_count(node) -> int:
        n = 1 if getattr(node, "run_type", None) == "tool" else 0
        for child in getattr(node, "child_runs", None) or []:
            n += _tool_count(child)
        return n

    outputs = getattr(run, "outputs", None) or {}
    answer = str(outputs.get("answer", outputs)).lower()
    required = ["severity", "disposition", "rationale", "containment", "uncertainty", "citation"]
    missing = [field for field in required if field not in answer]
    tools_used = _tool_count(run)
    passed = not missing and tools_used >= 2  # a grounded triage should consult fixtures, not answer from priors
    comment = f"missing sections: {missing or 'none'}; tool calls: {tools_used}"
    return {"key": "report_completeness_schema", "score": passed, "comment": comment}

## 4. Dry-run one example first

Score one example before scaling so the judge's comment is visible. `report_completeness_schema` is
trace-based, so it runs inside the full `evaluate()` call (it needs LangSmith's run tree), not here.

In [ ]:
outputs = run_agent(example["inputs"])
print("answer:", outputs["answer"][:240], "…\n")
print("severity_correctness:", severity_correctness_evaluator(example["inputs"], outputs, example["outputs"]))

## 5. Scale to the whole dataset → scorecard

`evaluate()` runs all 10 examples through the target, applies all evaluators, and returns iterable
results we aggregate locally before opening the experiment UI.

In [ ]:
from collections import defaultdict

from langsmith import evaluate

experiment = evaluate(
    run_agent,
    data=DATASET_NAME,
    evaluators=[
        severity_correctness_evaluator,       # LLM judge
        report_completeness_schema_evaluator,  # deterministic / trace-based
    ],
    experiment_prefix="alert-triage-02",
    max_concurrency=3,
)

scores: dict[str, list] = defaultdict(list)
for item in experiment:
    for r in item["evaluation_results"]["results"]:
        scores[r.key].append(r.score)

print("\n📊 Scorecard")
for key, vals in scores.items():
    if vals and all(isinstance(v, bool) for v in vals):
        print(f"  {key:30s}: {sum(vals)}/{len(vals)} passed")
    else:
        nums = [v for v in vals if isinstance(v, (int, float))]
        print(f"  {key:30s}: avg {sum(nums) / len(nums):.2f}" if nums else f"  {key}: n/a")

**Open the experiment in LangSmith** — per-example reports, judge comments, and traces behind every
verdict.

In [ ]:
try:
    from langsmith import Client

    print("🔗 experiment:", Client().read_project(project_name=experiment.experiment_name).url)
except Exception:
    pass  # evaluate() also prints a results URL; a missing link should never break the lab

## 6. Bonus — iterate & compare experiments

The point of an eval isn't one scorecard — it's the **loop**: change something, re-run, and compare
experiments to see if the change helped. That's how you improve an agent with evidence, not vibes.

```text
baseline experiment ──▶ change the agent (prompt / tools / model) ──▶ new experiment ──▶ compare in LangSmith
```

Try it yourself:
1. Edit the prompt — in the **Playground** (see Lab 01's bonus) or by tweaking `system_prompt` here.
2. Re-run `evaluate(...)` with a **new `experiment_prefix`** (e.g. `"alert-triage-02-v2"`), so it lands
   as a separate experiment:
   ```python
   agent_v2 = create_agent(model=model, tools=tools, system_prompt=my_edited_prompt, response_format=AlertTriageReport)
   def run_v2(inputs): return {"answer": agent_v2.invoke({"messages": inputs["messages"]})["structured_response"].model_dump_json()}
   evaluate(run_v2, data=DATASET_NAME, evaluators=[severity_correctness_evaluator, report_completeness_schema_evaluator], experiment_prefix="alert-triage-02-v2")
   ```
3. Open both experiments in LangSmith and use **Compare** to see per-example diffs and the score delta.

> 💡 On this small, clean dataset a change may move the score only a little (a strong model + the typed
> `response_format` already keep quality high). That's a real finding — **measure to confirm a change
> helped instead of assuming it did.** The bigger the eval set and the more borderline the cases, the
> more a genuine improvement will show.

## 7. Bonus — evaluator templates in LangSmith

You don't have to hand-write every judge. LangSmith ships **evaluator templates** — LLM-as-judge and
code evaluators — that you add from the UI, grouped by use case (Security, Safety, Quality,
Conversation, Trajectory…). Several map straight onto this lab: **PII Leakage**, **Prompt Injection**,
**Hallucination**, **Correctness** — the same concerns we grounded and guardrailed across Labs 01–03.

![LangSmith evaluator templates](images/02-langsmith-ui_evaluators.png)

On a dataset or experiment in LangSmith: **+ Evaluator → Create from a template** (or **Create from
scratch** for a custom LLM-judge / code evaluator), and browse **Show all templates** for the full
catalog. Add the few that fit your use case instead of writing every rubric from scratch.

## Recap

| Topic | API |
|---|---|
| Reference eval dataset | `seed_dataset()` / `seed_dataset(refresh=True)`, `dataset_url(...)` |
| Eval target (reused agent) | `create_agent(...)` → `run_agent(inputs)` |
| LLM-judge evaluator | structured-output LLM over `inputs, outputs, reference_outputs` |
| Deterministic evaluator | walk `run.child_runs`, inspect `run.outputs` |
| Scale + aggregate | `langsmith.evaluate(...)` → scorecard + experiment link |
| Iterate & compare | change the agent → new `experiment_prefix` → **Compare** in LangSmith |

**Next:** `03_orchestrate.ipynb` — add incident-thread memory, human approval, and LangGraph Studio.